In [1]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
from sklearn.pipeline import Pipeline
import json, os

df = pd.read_csv("../data/processed/cleaned.csv")
print("Total:", df.shape)

# smish SMS-এ keyword দেখুন
smish_df = df[df['label'] == 'smish']
print("\nSample smish SMS:")
for t in smish_df['text_clean'].head(5):
    print("-", t[:150])

Total: (7005, 4)

Sample smish SMS:
- আপনি ৩ vori gold জিতেছেন! Claim করুন: [PHONE]
- Apnar payment-ta prokria korte amader apnar tothyo proyojon. Ekhane click korun: [URL]
- জয়! আপনি Google লটারিতে $৫০০ জিতেছেন। claim করতে: winlottery.tk/claim পূরণ করুন।
- আপনার debit card-এর limit increase করতে click করুন: [URL]
- বাংলাদেশ ব্যাংক থেকে গুরুত্বপূর্ণ বার্তা। বিস্তারিত জানতে এখানে ক্লিক করুন: [URL]


In [2]:
# smish SMS-কে ২টি group-এ ভাগ করুন: URL থাকা vs না থাকা
df['has_url'] = df['text_clean'].str.contains(r'\[URL\]', regex=True)
df['has_phone'] = df['text_clean'].str.contains(r'\[PHONE\]', regex=True)

print("Smish with URL:", ((df['label']=='smish') & df['has_url']).sum())
print("Smish with phone:", ((df['label']=='smish') & df['has_phone']).sum())
print("Smish with neither:", ((df['label']=='smish') & ~df['has_url'] & ~df['has_phone']).sum())

Smish with URL: 1542
Smish with phone: 806
Smish with neither: 466


In [3]:
# ট্রেনিং: smish যেগুলোতে URL আছে
# টেস্ট  : smish যেগুলোতে URL নেই (unseen attack type)
train_hard = df[~((df['label']=='smish') & (~df['has_url']))].reset_index(drop=True)
test_hard  = df[((df['label']=='smish') & (~df['has_url']))].reset_index(drop=True)

print("Train hard:", train_hard.shape, "| label dist:", train_hard['label'].value_counts().to_dict())
print("Test hard :", test_hard.shape, "| label dist:", test_hard['label'].value_counts().to_dict())

train_hard.to_csv("../data/processed/train_hard.csv", index=False)
test_hard.to_csv("../data/processed/test_hard.csv", index=False)

Train hard: (5738, 6) | label dist: {'normal': 2488, 'promo': 1708, 'smish': 1542}
Test hard : (1267, 6) | label dist: {'smish': 1267}


In [4]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
from sklearn.pipeline import Pipeline

df = pd.read_csv("../data/processed/cleaned.csv")

# URL ও PHONE চিহ্নিত করুন
df['has_url']   = df['text_clean'].str.contains(r'\[URL\]', regex=True)
df['has_phone'] = df['text_clean'].str.contains(r'\[PHONE\]', regex=True)

# ---------- Strategy B1: URL-Holdout ----------
# Train: URL-হীন smish বাদ, বাকি সব রাখা (normal + promo + URL-সহ smish)
# Test : URL-হীন smish + normal + promo (সম্পূর্ণ নতুন attack type)

unseen_smish = df[(df['label'] == 'smish') & (~df['has_url'])].reset_index(drop=True)
seen_data    = df[~((df['label'] == 'smish') & (~df['has_url']))].reset_index(drop=True)

# Test বানান: unseen_smish + normal + promo থেকে sample
from sklearn.model_selection import train_test_split

# প্রথমে seen_data থেকে ২০% normal ও promo টেস্টে নিন
seen_smish   = seen_data[seen_data['label'] == 'smish']
seen_normal  = seen_data[seen_data['label'] == 'normal']
seen_promo   = seen_data[seen_data['label'] == 'promo']

# Train / Test-এর জন্য normal ও promo ভাগ করুন
normal_train, normal_test = train_test_split(seen_normal, test_size=0.2, random_state=42)
promo_train, promo_test   = train_test_split(seen_promo,  test_size=0.2, random_state=42)
smish_train = seen_smish  # URL-সহ smish সব ট্রেনিং-এ

# Train: ৩টি ক্লাসই (কিন্তু smish-এর মধ্যে শুধু URL-সহ)
train_hard2 = pd.concat([smish_train, normal_train, promo_train], ignore_index=True)

# Test: URL-হীন smish (unseen) + normal + promo
test_hard2  = pd.concat([unseen_smish, normal_test, promo_test], ignore_index=True)

print("=" * 60)
print("Strategy B1: URL-Holdout")
print("=" * 60)
print(f"Train: {train_hard2.shape}")
print(train_hard2['label'].value_counts())
print(f"\nTest: {test_hard2.shape}")
print(test_hard2['label'].value_counts())

# Baseline চালান
def build_baseline():
    return Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.9, sublinear_tf=True)),
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
    ])

clf = build_baseline()
clf.fit(train_hard2['text_clean'], train_hard2['label'])
pred = clf.predict(test_hard2['text_clean'])

f1_b1 = f1_score(test_hard2['label'], pred, average='macro')
print(f"\nMacro F1: {f1_b1:.4f}")
print(classification_report(test_hard2['label'], pred))

# সেভ
train_hard2.to_csv("../data/processed/train_hard2.csv", index=False)
test_hard2.to_csv("../data/processed/test_hard2.csv", index=False)

Strategy B1: URL-Holdout
Train: (4898, 6)
label
normal    1990
smish     1542
promo     1366
Name: count, dtype: int64

Test: (2107, 6)
label
smish     1267
normal     498
promo      342
Name: count, dtype: int64

Macro F1: 0.5963
              precision    recall  f1-score   support

      normal       0.38      0.98      0.55       498
       promo       0.71      0.96      0.81       342
       smish       0.99      0.27      0.43      1267

    accuracy                           0.55      2107
   macro avg       0.69      0.74      0.60      2107
weighted avg       0.80      0.55      0.52      2107



In [5]:
# Train: ফোন-হীন smish বাদ
# Test : ফোন-হীন smish (unseen) + normal + promo

unseen_smish_p = df[(df['label'] == 'smish') & (~df['has_phone'])].reset_index(drop=True)
seen_data_p    = df[~((df['label'] == 'smish') & (~df['has_phone']))].reset_index(drop=True)

print("Unseen smish (no phone):", len(unseen_smish_p))
print("Seen data size:", len(seen_data_p))

# একই লজিক
smish_train_p  = seen_data_p[seen_data_p['label'] == 'smish']
normal_p       = seen_data_p[seen_data_p['label'] == 'normal']
promo_p        = seen_data_p[seen_data_p['label'] == 'promo']

normal_train_p, normal_test_p = train_test_split(normal_p, test_size=0.2, random_state=42)
promo_train_p, promo_test_p   = train_test_split(promo_p,  test_size=0.2, random_state=42)

train_hard3 = pd.concat([smish_train_p, normal_train_p, promo_train_p], ignore_index=True)
test_hard3  = pd.concat([unseen_smish_p, normal_test_p, promo_test_p], ignore_index=True)

print("\nStrategy B2: Phone-Holdout")
print(f"Train: {train_hard3.shape}")
print(train_hard3['label'].value_counts())
print(f"\nTest: {test_hard3.shape}")
print(test_hard3['label'].value_counts())

clf = build_baseline()
clf.fit(train_hard3['text_clean'], train_hard3['label'])
pred = clf.predict(test_hard3['text_clean'])
f1_b2 = f1_score(test_hard3['label'], pred, average='macro')
print(f"\nMacro F1: {f1_b2:.4f}")
print(classification_report(test_hard3['label'], pred))

train_hard3.to_csv("../data/processed/train_hard3.csv", index=False)
test_hard3.to_csv("../data/processed/test_hard3.csv", index=False)

Unseen smish (no phone): 2003
Seen data size: 5002

Strategy B2: Phone-Holdout
Train: (4162, 6)
label
normal    1990
promo     1366
smish      806
Name: count, dtype: int64

Test: (2843, 6)
label
smish     2003
normal     498
promo      342
Name: count, dtype: int64

Macro F1: 0.5329
              precision    recall  f1-score   support

      normal       0.54      0.97      0.69       498
       promo       0.26      0.96      0.41       342
       smish       0.98      0.33      0.50      2003

    accuracy                           0.52      2843
   macro avg       0.59      0.75      0.53      2843
weighted avg       0.81      0.52      0.52      2843



In [ ]:
import os
import pandas as pd
from sklearn.metrics import f1_score
import json

def build_baseline():
    return Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.9, sublinear_tf=True)),
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
    ])

def eval_split(tr_path, te_path, name):
    if not os.path.exists(tr_path) or not os.path.exists(te_path):
        return None
    tr = pd.read_csv(tr_path)
    te = pd.read_csv(te_path)
    clf = build_baseline()
    clf.fit(tr['text_clean'], tr['label'])
    pred = clf.predict(te['text_clean'])
    f1 = f1_score(te['label'], pred, average='macro')
    print(f"{name:35s} | Train: {len(tr):5d} | Test: {len(te):5d} | Macro F1: {f1:.4f}")
    return f1

print("=" * 90)
print(f"{'SPLIT':35s} | {'TRAIN':>5s} | {'TEST':>5s} | {'Macro F1':>8s}")
print("=" * 90)

results = {}
results['ID (baseline)']       = eval_split("../data/processed/train_id.csv",  "../data/processed/test_id.csv",  "ID Split")
results['OOD (old)']           = eval_split("../data/processed/train_ood.csv", "../data/processed/test_ood.csv", "OOD — Banglish + CodeMix")
results['B1: URL Holdout']     = eval_split("../data/processed/train_hard2.csv","../data/processed/test_hard2.csv","B1: URL Holdout (unseen smish)")
results['B2: Phone Holdout']   = eval_split("../data/processed/train_hard3.csv","../data/processed/test_hard3.csv","B2: Phone Holdout (unseen smish)")
results['C: Bengali→CodeMix']  = eval_split("../data/processed/train_cm.csv",  "../data/processed/test_cm.csv",  "C: Bengali→CodeMix")

print("=" * 90)

# Gap হিসাব
if results['ID (baseline)']:
    print("\nGeneralization Gap (vs ID):")
    for k, v in results.items():
        if v is not None and k != 'ID (baseline)':
            gap = results['ID (baseline)'] - v
            flag = "✅" if gap > 0.05 else "⚠️"
            print(f"  {flag} {k:30s} gap = {gap:.4f}")

# সেভ
os.makedirs("../results", exist_ok=True)
with open("../results/hard_split_results.json", "w") as f:
    json.dump({k: float(v) for k, v in results.items() if v is not None}, f, indent=2)
print("\nSaved: ../results/hard_split_results.json")

SPLIT                               | TRAIN |  TEST | Macro F1
ID Split                            | Train:  5604 | Test:  1401 | Macro F1: 0.9702
OOD — Banglish + CodeMix            | Train:  3525 | Test:  3480 | Macro F1: 0.9472
B1: URL Holdout (unseen smish)      | Train:  4898 | Test:  2107 | Macro F1: 0.5963
B2: Phone Holdout (unseen smish)    | Train:  4162 | Test:  2843 | Macro F1: 0.5329

Generalization Gap (vs ID):
  ⚠️ OOD (old)                      gap = 0.0230
  ✅ B1: URL Holdout                gap = 0.3738
  ✅ B2: Phone Holdout              gap = 0.4373

Saved: ../results/hard_split_results.json


: 